# Chapter 4: Trees & Forests — Decision-Based Models

Decision trees make predictions by asking a series of **yes/no questions**.

```
Is it raining?
├── Yes → Is wind > 30mph?
│   ├── Yes → Stay home
│   └── No  → Take umbrella
└── No  → Go outside
```

The model **learns which questions to ask and in what order** from the data.

---
## Decision Tree: How It Works

At each step, the tree picks the feature and threshold that **best separates** the classes.

"Best" means: after the split, each group is as pure (one class) as possible.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Should I play tennis today?
# Features: [outlook, temperature, humidity, wind]
# outlook:  0=sunny, 1=overcast, 2=rainy
# wind:     0=weak, 1=strong
data = np.array([
    [0, 85, 85, 0],   # sunny, hot, high humidity, weak wind → No
    [0, 80, 90, 1],   # sunny, hot, high humidity, strong   → No
    [1, 83, 86, 0],   # overcast, hot                       → Yes
    [2, 70, 96, 0],   # rainy, mild                         → Yes
    [2, 68, 80, 0],   # rainy, cool                         → Yes
    [2, 65, 70, 1],   # rainy, cool, strong wind            → No
    [1, 64, 65, 1],   # overcast, cool                      → Yes
    [0, 72, 95, 0],   # sunny, mild, high humidity           → No
    [0, 69, 70, 0],   # sunny, cool, normal humidity         → Yes
    [2, 75, 80, 0],   # rainy, mild                         → Yes
    [0, 75, 70, 1],   # sunny, mild, normal, strong         → Yes
    [1, 72, 90, 1],   # overcast                            → Yes
    [1, 81, 75, 0],   # overcast, hot                       → Yes
    [2, 71, 91, 1],   # rainy, mild, strong wind            → No
])
play = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0])  # 0=No, 1=Yes

feature_names = ["outlook", "temperature", "humidity", "wind"]

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(data, play)

print("Decision Tree learned these rules:\n")
print(export_text(tree, feature_names=feature_names))
print("The tree figured out: humidity and wind matter most!")

---
## Feature Importance: Which Features Matter?

Decision trees can tell us how important each feature is for predictions.

In [ ]:
print("Feature Importance (how much each feature contributes):")
print()
for name, importance in sorted(zip(feature_names, tree.feature_importances_), 
                                key=lambda x: -x[1]):
    bar = "█" * int(importance * 40)
    print(f"  {name:<15} {importance:>6.1%}  {bar}")

print("\nThe tree discovered which features split the data best.")
print("We didn't tell it — it learned from the data.")

---
## Tree Depth: Simple vs Complex

- **Shallow tree** (depth=2): few questions, may underfit
- **Deep tree** (depth=20): many questions, may overfit (memorize noise)

This is the same bias-variance tradeoff from regression.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

print("Effect of Tree Depth on Iris Classification:")
print(f"{'Depth':>6} {'Train':>8} {'Test':>8} {'Gap':>6} {'Assessment':>15}")
print(f"{'─'*6} {'─'*8} {'─'*8} {'─'*6} {'─'*15}")

for depth in [1, 2, 3, 4, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    train_acc = t.score(X_train, y_train)
    test_acc = t.score(X_test, y_test)
    gap = train_acc - test_acc
    
    depth_str = str(depth) if depth else "None"
    if depth == 1:
        assess = "underfit"
    elif depth and depth <= 4:
        assess = "good"
    else:
        assess = "overfit risk"
    
    print(f"{depth_str:>6} {train_acc:>8.1%} {test_acc:>8.1%} {gap:>+6.1%} {assess:>15}")

print("\nBig gap between train and test = overfitting")
print("'None' means no limit → tree grows until every leaf is pure")

---
## Random Forest: Many Trees > One Tree

One tree can be noisy and overfit. Solution: build MANY trees and let them **vote**.

```
Tree 1: "I think it's class A"
Tree 2: "I think it's class B"
Tree 3: "I think it's class A"
Tree 4: "I think it's class A"
Tree 5: "I think it's class B"

Vote: 3 for A, 2 for B → Final answer: A
```

Each tree is trained on a **random subset** of the data and features. This diversity makes the ensemble stronger than any individual tree.

This is called **ensemble learning** — combining weak models to make a strong one.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Compare single tree vs forest
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

print("Single Tree vs Random Forest (100 trees):")
print(f"{'Model':<20} {'Train':>8} {'Test':>8}")
print(f"{'─'*20} {'─'*8} {'─'*8}")
print(f"{'Single Tree':<20} {single_tree.score(X_train, y_train):>8.1%} {single_tree.score(X_test, y_test):>8.1%}")
print(f"{'Random Forest':<20} {forest.score(X_train, y_train):>8.1%} {forest.score(X_test, y_test):>8.1%}")
print()

# Show how individual trees vote
sample = X_test[0:1]
actual = iris.target_names[y_test[0]]
print(f"Sample prediction for: {actual} (actual)")
print(f"How 10 individual trees vote:")
votes = {name: 0 for name in iris.target_names}
for i, tree in enumerate(forest.estimators_[:10]):
    pred = iris.target_names[tree.predict(sample)[0]]
    votes[pred] += 1
    print(f"  Tree {i+1:>2}: {pred}")
print(f"\nVotes: {dict(votes)}")
print(f"Forest prediction: {iris.target_names[forest.predict(sample)[0]]}")

In [ ]:
# How many trees do you need?
print("Effect of Number of Trees:")
print(f"{'Trees':>6} {'Test Acc':>10}")
print(f"{'─'*6} {'─'*10}")

for n in [1, 5, 10, 25, 50, 100, 200, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    bar = "█" * int(acc * 30)
    print(f"{n:>6} {acc:>10.1%}  {bar}")

print("\nDiminishing returns: 100 trees is usually enough.")
print("More trees = more computation but barely more accuracy.")

---
## Gradient Boosting: Trees That Learn from Mistakes

Random Forest: many trees in **parallel** (independent).
Gradient Boosting: many trees in **sequence** (each fixes previous errors).

```
Tree 1: makes predictions (some wrong)
Tree 2: focuses on what Tree 1 got wrong
Tree 3: focuses on what Tree 1+2 still get wrong
...
Final = sum of all trees
```

Often the most accurate method for structured/tabular data.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Compare all tree-based methods
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print("Tree-Based Model Comparison:")
print(f"{'Model':<22} {'Train':>8} {'Test':>8}")
print(f"{'─'*22} {'─'*8} {'─'*8}")

for name, m in models.items():
    m.fit(X_train, y_train)
    train_acc = m.score(X_train, y_train)
    test_acc = m.score(X_test, y_test)
    print(f"{name:<22} {train_acc:>8.1%} {test_acc:>8.1%}")

print("\nGradient Boosting often wins on structured data.")
print("(XGBoost, LightGBM are popular optimized versions)")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Decision Tree** | Series of yes/no questions learned from data |
| **Feature Importance** | Tree tells you which features matter most |
| **Tree Depth** | Deeper = more complex, risk of overfitting |
| **Random Forest** | Many independent trees vote → more stable |
| **Gradient Boosting** | Trees in sequence, each fixes previous errors |
| **Ensemble** | Combining weak models makes a strong one |

**Next chapter**: Clustering — finding groups without labels